In [ ]:
# =============================================================================
# jobs/14_fgs_constituency_contacts.py
# Link the FGS (the forecast) to constituencies and MPs: the PROACTIVE view.
#
# 13_warning_constituency_contacts is the reactive view: it starts from warnings
# the EA has already issued. This notebook starts from the Flood Guidance
# Statement, the forecast of what may come in the next five days, so an MP can
# be told before any warning exists.
#
# What this script does:
#   1. Picks the latest statement that has been fully intersected (it has an
#      up-to-date row in fgs_intersections_log). The table is the CURRENT
#      forecast, not history: MP details are overwritten weekly, so stamping old
#      forecasts with today's MP would mislead.
#   2. Takes that statement's rows from fgs_constituency_intersections (written
#      by 07): one per FGS geometry, forecast day, flood source and constituency.
#   3. Attaches the sitting MP and contact details (mp_contact_details, from 04).
#   4. Overwrites fgs_constituency_mp_contacts in full.
#
# GRAIN: one row per (FGS geometry, forecast day, source, constituency).
# Several rows per constituency and day are normal, because a constituency can
# sit under several geometries and sources. To get the worst risk per
# constituency, day and source, rank on risk_rank (see the README query).
#
# risk_rank: 1 Very Low (Green), 2 Low (Yellow), 3 Medium (Amber), 4 High (Red),
# 0 unknown. It is derived from risk_level so you can sort and filter on a number.
#
# FGS_CONTACT_MIN_PCT (config.py) drops polygons that cover less than that share
# of the CONSTITUENCY. Coastal polylines have no area (share_of_constituency_covered
# is NULL) and are always kept.
#
# NO SPATIAL WORK HAPPENS HERE: the geometry was done in 07. This is plain Spark
# SQL, cheap enough to run on every poll, straight after 07.
#
# Exit values: "success".
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

from config import (
    TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_CONST_INTERSECT,
    TBL_MP_CONTACTS, TBL_FGS_CONTACTS, FGS_CONTACT_MIN_PCT
)
from utils.helpers import get_spark, utc_now, table_exists, constituency_name_key


# =============================================================================
# SETUP
# =============================================================================

spark   = get_spark()
now_iso = utc_now().isoformat()
name_key = constituency_name_key

for table in (TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_CONST_INTERSECT, TBL_MP_CONTACTS):
    if not table_exists(spark, table):
        raise Exception(
            f"{table} does not exist. Run 01/02, 07_compute_intersections and "
            f"04_refresh_mp_details first."
        )


In [ ]:
# =============================================================================
# CHOOSE THE STATEMENT
# The latest statement whose intersections are complete and up to date. This is
# normally the newest statement. If 07 has not yet caught up with a new or
# modified statement, the table keeps showing the previous complete one rather
# than a half-built picture, and says so.
# =============================================================================

complete = spark.sql(f"""
    SELECT l.statement_id,
           l.last_modified_at,
           l.computed_at AS intersections_computed_at,
           s.issued_at
    FROM {TBL_FGS_INTERSECT_LOG} l
    JOIN {TBL_STATEMENTS} s
      ON s.statement_id = l.statement_id
     AND COALESCE(s.last_modified_at, '') = COALESCE(l.last_modified_at, '')
    ORDER BY s.issued_at DESC, l.statement_id DESC
    LIMIT 1
""").collect()

if not complete:
    raise Exception(
        "No statement has up-to-date intersections. Run 07_compute_intersections first."
    )

chosen        = complete[0]
statement_id  = int(chosen["statement_id"])
last_modified = (chosen["last_modified_at"] or "").replace("'", "''")   # for the SQL below

newest = spark.sql(f"""
    SELECT statement_id FROM {TBL_STATEMENTS}
    ORDER BY issued_at DESC, statement_id DESC LIMIT 1
""").collect()[0]["statement_id"]

print(f"Using statement {statement_id} (issued {chosen['issued_at']}).")
if int(newest) != statement_id:
    print(f"WARNING: statement {newest} is newer but its intersections are not "
          f"complete yet. Showing statement {statement_id} until 07 catches up.")


In [ ]:
# =============================================================================
# BUILD THE TABLE
# =============================================================================

query = f"""
WITH mp AS (
    -- One MP per constituency key, so a duplicate name cannot fan rows out.
    SELECT * FROM (
        SELECT
            m.*,
            {name_key('m.constituency_name')} AS name_key,
            ROW_NUMBER() OVER (
                PARTITION BY {name_key('m.constituency_name')}
                ORDER BY m.member_id
            ) AS rn
        FROM {TBL_MP_CONTACTS} m
    ) ranked
    WHERE rn = 1
)
SELECT
    -- the forecast
    i.statement_id,
    i.issued_at,
    '{last_modified}'                         AS statement_last_modified_at,
    i.poly_id,
    i.day_index,
    i.forecast_date,
    i.source,
    i.risk_x,
    i.risk_y,
    i.impact_label,
    i.likelihood_label,
    i.risk_level,
    i.risk_colour,
    CASE i.risk_level
        WHEN 'Very Low' THEN 1
        WHEN 'Low'      THEN 2
        WHEN 'Medium'   THEN 3
        WHEN 'High'     THEN 4
        ELSE 0
    END                                       AS risk_rank,
    i.geometry_type,
    -- the constituency
    i.constituency_id,
    i.constituency_name,
    i.intersection_pct                        AS share_of_constituency_covered,
    -- the MP
    (mp.member_id IS NOT NULL)                AS mp_matched,
    mp.member_id                              AS mp_member_id,
    mp.name                                   AS mp_name,
    mp.party                                  AS mp_party,
    mp.email                                  AS mp_email,
    mp.phone                                  AS mp_phone,
    mp.website                                AS mp_website,
    mp.twitter_handle                         AS mp_twitter,
    -- audit
    i.computed_at                             AS intersections_computed_at,
    '{now_iso}'                               AS computed_at
FROM {TBL_FGS_CONST_INTERSECT} i
LEFT JOIN mp
  ON mp.name_key = {name_key('i.constituency_name')}
WHERE i.statement_id = {statement_id}
  AND (i.intersection_pct IS NULL OR i.intersection_pct >= {FGS_CONTACT_MIN_PCT})
"""

result = spark.sql(query).cache()
n_rows = result.count()
n_const = result.select("constituency_id").distinct().count()
print(f"Rows: {n_rows}   constituencies: {n_const}")

if n_rows == 0:
    print("No constituency intersections for this statement (a low-risk day, or "
          "every overlap is below FGS_CONTACT_MIN_PCT). The table will be empty.")

if n_rows > 0:
    n_matched = result.filter("mp_matched").select("constituency_id").distinct().count()
    if n_matched == 0:
        # Every constituency failed to match an MP: the two name sources have
        # stopped agreeing, or the MP table is empty.
        raise Exception(
            "No constituency matched an MP on name. Check mp_contact_details and "
            "parliamentary_constituencies."
        )
    no_mp = result.filter("NOT mp_matched").select("constituency_name").distinct().collect()
    if no_mp:
        print(f"\nNOTE: {len(no_mp)} constituenc(ies) have no matching MP "
              f"(vacancy, or a name that does not match). First 10:")
        for r in no_mp[:10]:
            print(f"  {r['constituency_name']}")


In [ ]:
# =============================================================================
# WRITE TO DELTA
# Full overwrite: this table is the CURRENT forecast, rebuilt every poll.
# =============================================================================

(
    result.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TBL_FGS_CONTACTS)
)

spark.sql(f"""
    COMMENT ON TABLE {TBL_FGS_CONTACTS} IS
    'The latest complete FGS forecast, one row per FGS geometry, forecast day, flood
     source and constituency it overlaps, with the sitting MP and contact details.
     The proactive counterpart of flood_warning_constituency_contacts. Rebuilt on
     every poll by 14_fgs_constituency_contacts from fgs_constituency_intersections
     and mp_contact_details. risk_rank: 1 Very Low, 2 Low, 3 Medium, 4 High.
     mp_matched = false marks a constituency with no matching MP.'
""")

written = spark.table(TBL_FGS_CONTACTS).count()
print(f"\nVerified: {written} rows written to {TBL_FGS_CONTACTS}.")
result.unpersist()

dbutils.notebook.exit("success")
